# ComfyUI＋LTX-2.3 個人テスト用ノートブック（改修版）

下の【起動】セルの ▶ を1回押すだけで動きます。

- **GPU（A100）で接続した場合**：準備 → 導入 → 起動 → 画面のリンク表示 → 見張り、まで自動で進みます
- **CPU で接続した場合**：モデルのダウンロード（初回だけ、約43GB）だけ行って止まります

外部へデータを送らない設定（`--offline`）でComfyUIを起動します。画面はColab経由で開き、中継サービスは使いません。

In [ ]:
# 【起動】このセルの ▶ を押すだけ。動き続けるのが正常です（止めないでください）

# ===== 設定（必要なら数字を変える） =====
IDLE_MINUTES = 30      # 生成していない時間がこの分数続いたら自動で切断
MAX_HOURS = 4          # 起動からこの時間が過ぎたら自動で切断
COPY_TO_LOCAL = True   # モデルをColab内の速いディスクにコピーしてから使う
# =======================================

import os, shutil, socket, subprocess, time, requests
from IPython import get_ipython
from google.colab import drive, output, runtime

def step(msg):
    print(f'\n■ {msg}', flush=True)

# --- Driveの接続と保存先 ---
step('Google Driveに接続')
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/ComfyUI_LTX'
DRIVE_MODELS, OUTPUT, USERDIR = f'{BASE}/models', f'{BASE}/output', f'{BASE}/user'
SUBS = ['checkpoints', 'loras', 'text_encoders', 'latent_upscale_models']
for d in [f'{DRIVE_MODELS}/{s}' for s in SUBS] + [OUTPUT, USERDIR]:
    os.makedirs(d, exist_ok=True)

# --- モデルの確認（足りなければダウンロード） ---
# ComfyUI公式テンプレート「LTX-2.3 Text to Video」が指定しているファイル
MODEL_FILES = [
    ('checkpoints',           'https://huggingface.co/Lightricks/LTX-2.3-fp8/resolve/main/ltx-2.3-22b-dev-fp8.safetensors'),
    ('loras',                 'https://huggingface.co/Comfy-Org/ltx-2.3/resolve/main/split_files/loras/ltx_2.3_22b_distilled_1.1_lora_dynamic_fro09_avg_rank_111_bf16.safetensors'),
    ('loras',                 'https://huggingface.co/Comfy-Org/ltx-2/resolve/main/split_files/loras/gemma-3-12b-it-abliterated_lora_rank64_bf16.safetensors'),
    ('text_encoders',         'https://huggingface.co/Comfy-Org/ltx-2/resolve/main/split_files/text_encoders/gemma_3_12B_it_fp4_mixed.safetensors'),
    ('latent_upscale_models', 'https://huggingface.co/Lightricks/LTX-2.3/resolve/main/ltx-2.3-spatial-upscaler-x2-1.1.safetensors'),
]

def expected_size(url):
    try:
        return int(requests.head(url, allow_redirects=True, timeout=60).headers.get('content-length', 0))
    except Exception:
        return 0

def is_done(path, size):
    if not os.path.exists(path):
        return False
    return os.path.getsize(path) == size if size else os.path.getsize(path) > 1_000_000

step('モデルの確認')
all_ok = True
for sub, url in MODEL_FILES:
    name = url.split('/')[-1]
    dest = f'{DRIVE_MODELS}/{sub}/{name}'
    size = expected_size(url)
    if is_done(dest, size):
        print(f'  済み: {name}')
        continue
    print(f'  ダウンロード中: {name}（{size / 1e9:.1f} GB）')
    get_ipython().system(f'wget -c -q --show-progress -O "{dest}" "{url}"')
    ok = is_done(dest, size)
    all_ok = all_ok and ok
    print('  → 完了' if ok else '  → 未完了')
if not all_ok:
    raise SystemExit('未完了のファイルがあります。もう一度 ▶ を押すと続きから再開します')

# --- GPUがなければここで終わり ---
if not shutil.which('nvidia-smi'):
    print('\nGPUなし（CPU）で接続しています。モデルはそろいました。')
    print('動画を作るときは「ランタイム」→「ランタイムのタイプを変更」でA100を選び、もう一度 ▶ を押してください。')
    raise SystemExit
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                     capture_output=True, text=True).stdout)

# --- ComfyUIの導入 ---
step('ComfyUIの導入（3〜5分）')
if not os.path.exists('/content/ComfyUI'):
    get_ipython().system('git clone -q --depth 1 https://github.com/Comfy-Org/ComfyUI /content/ComfyUI')
get_ipython().system('pip install -q -r /content/ComfyUI/requirements.txt')

# --- モデルを速いディスクへコピー ---
MODELS = DRIVE_MODELS
if COPY_TO_LOCAL:
    step('モデルをColab内のディスクへコピー（約43GB）')
    t0 = time.time()
    LOCAL = '/content/models'
    for sub, url in MODEL_FILES:
        name = url.split('/')[-1]
        src, dst = f'{DRIVE_MODELS}/{sub}/{name}', f'{LOCAL}/{sub}/{name}'
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        if not (os.path.exists(dst) and os.path.getsize(dst) == os.path.getsize(src)):
            print(f'  コピー中: {name}', flush=True)
            shutil.copyfile(src, dst)
    MODELS = LOCAL
    print(f'  コピー時間: {(time.time() - t0) / 60:.1f} 分')

# --- ComfyUIの起動 ---
step('ComfyUIを起動（1〜3分）')
subprocess.run(['pkill', '-f', 'main.py --offline'])  # 動いている古いComfyUIを止める
proc = subprocess.Popen(
    ['python', '/content/ComfyUI/main.py',
     '--offline',                                               # 外部との通信と有料API部品を止める
     '--enable-cors-header', 'https://colab.research.google.com',  # Colab経由の表示を許可（ないと403で真っ白）
     '--listen', '127.0.0.1', '--port', '8188',
     '--models-directory', MODELS,
     '--output-directory', OUTPUT,
     '--user-directory', USERDIR],
    cwd='/content/ComfyUI', stdout=open('/content/comfyui.log', 'w'), stderr=subprocess.STDOUT,
    start_new_session=True)  # ■でこのセルを止めてもComfyUIは止まらない

for _ in range(600):
    if proc.poll() is not None:
        break
    with socket.socket() as s:
        if s.connect_ex(('127.0.0.1', 8188)) == 0:
            break
    time.sleep(1)
if proc.poll() is not None:
    print('起動に失敗しました。ログの最後：')
    print(open('/content/comfyui.log').read()[-3000:])
    raise SystemExit

# --- 画面を開く ---
step('起動しました。下のリンクを押すと、新しいタブにComfyUIの画面が開きます')
output.serve_kernel_port_as_window(8188, anchor_text='▶ ComfyUIを新しいタブで開く', skip_warning=True)
print('\n（新しいタブが真っ白の場合は、この下のColab内の画面を使ってください）')
output.serve_kernel_port_as_iframe(8188, height='900')

# --- 見張り（自動切断） ---
def stop_runtime(reason):
    print(reason + ' → Driveへの書き込みを終えてから切断します', flush=True)
    try:
        drive.flush_and_unmount()
    except Exception:
        pass
    runtime.unassign()

start, idle = time.time(), 0
try:
    while True:
        time.sleep(60)
        if proc.poll() is not None:
            print('ComfyUIが停止しました。ログ: /content/comfyui.log')
            break
        try:
            q = requests.get('http://127.0.0.1:8188/queue', timeout=10).json()
            busy = len(q.get('queue_running', [])) + len(q.get('queue_pending', [])) > 0
        except Exception:
            busy = True  # 応答がないのは処理中とみなす
        idle = 0 if busy else idle + 1
        if idle >= IDLE_MINUTES:
            stop_runtime(f'{IDLE_MINUTES}分間生成がありませんでした')
            break
        if (time.time() - start) / 3600 >= MAX_HOURS:
            stop_runtime(f'起動から{MAX_HOURS}時間が過ぎました')
            break
except KeyboardInterrupt:
    print('見張りを止めました（ComfyUIは動いたままです）。使い終わったら必ず「ランタイム」→「ランタイムを接続解除して削除」を押してください')

## 補助：進み具合の確認（必要なときだけ）

【起動】セルが動いている間は他のセルを実行できません。確認したいときは、【起動】セルの ■ を押して見張りを止めてから実行します（ComfyUIは止まりません。その後は手動で切断してください）。

In [ ]:
# 【進み具合の確認】30秒ごとにGPUとログの最後の1行を10分間表示
import subprocess, time
for i in range(20):
    mem = subprocess.run(['nvidia-smi', '--query-gpu=utilization.gpu,memory.used', '--format=csv,noheader'],
                         capture_output=True, text=True).stdout.strip()
    last = open('/content/comfyui.log').read().strip().splitlines()[-1][-120:]
    print(f'{i*30:>4}秒  GPU {mem}  |  {last}')
    time.sleep(30)